# Ch.2 — Non-Linear Algebra: Polynomials and the Feature-Expansion Trick

**Running theme.** Gravity bends the basketball's path into a parabola. The whole trajectory is $y(t) = v_{0y}\,t - \tfrac{1}{2}g\,t^2$ — non-linear in $t$. And yet, we'll fit it with Ch.1's linear machinery by inventing one extra feature.

**Learning outcomes.** After this notebook you will:
1. Drag the three parabola coefficients and watch the curve reshape.
2. Fit a full free-throw trajectory using `LinearRegression` + `PolynomialFeatures`.
3. See the parabola $y = a x^2 + b x + c$ as a *flat plane* when you plot it in $(x_1, x_2)$ space with $x_1 = x^2$.
4. Recognise when feature expansion works — and when it doesn't.

## 1 · Imports

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from ipywidgets import FloatSlider, FloatText, HBox, VBox, Output, jslink
from sklearn.linear_model import LinearRegression
from sklearn.preprocessing import PolynomialFeatures

try:
    get_ipython().run_line_magic("matplotlib", "widget")
except Exception:
    get_ipython().run_line_magic("matplotlib", "inline")

plt.rcParams.update({"figure.figsize": (8.5, 5.0), "figure.dpi": 110,
                     "axes.grid": True, "grid.alpha": 0.25})
np.random.seed(2)

def linked_pair(label, value, vmin, vmax, step=0.05):
    """Slider + text-box linked bidirectionally."""
    sl = FloatSlider(value=value, min=vmin, max=vmax, step=step,
                     description=label, continuous_update=True, readout=False)
    tx = FloatText(value=value, description=" ", step=step,
                   layout={"width": "110px"})
    jslink((sl, "value"), (tx, "value"))
    return sl, tx

## 2 · Drag the parabola coefficients

The equation $y = a\,x^2 + b\,x + c$ has three dials:

- **$a$** sets *curvature*. Positive $a$ opens upward, negative opens downward, $|a|$ controls how tight.
- **$b$** shifts the *vertex* sideways. Vertex is at $x = -b/(2a)$.
- **$c$** is the *$y$-intercept* — value when $x = 0$.

Experiment to develop intuition. Then predict: what's the vertex $x$ for $a = 1, b = -4$? (Answer below.)

In [ ]:
a_sl, a_tx = linked_pair("a", value=0.8,  vmin=-2.0, vmax=2.0, step=0.05)
b_sl, b_tx = linked_pair("b", value=-1.2, vmin=-4.0, vmax=4.0, step=0.05)
c_sl, c_tx = linked_pair("c", value=1.0,  vmin=-3.0, vmax=3.0, step=0.05)
info_out = Output()

xs = np.linspace(-4, 4, 300)
fig, ax = plt.subplots()
parab_line, = ax.plot(xs, a_sl.value * xs**2 + b_sl.value * xs + c_sl.value,
                      color="#8E44AD", lw=2.4)
y_int  = ax.scatter([0], [c_sl.value], color="#F39C12", s=60, zorder=5)
vx     = -b_sl.value / (2 * a_sl.value) if a_sl.value != 0 else 0.0
vy     = a_sl.value * vx**2 + b_sl.value * vx + c_sl.value
vertex = ax.scatter([vx], [vy], color="#E74C3C", s=60, zorder=5)
ax.axhline(0, color="#7F8C8D", lw=0.6); ax.axvline(0, color="#7F8C8D", lw=0.6)
ax.set_xlim(-4, 4); ax.set_ylim(-6, 12)
ax.set_xlabel("x"); ax.set_ylabel("y = a·x² + b·x + c")

def redraw_parab(*_):
    a, b, c = a_sl.value, b_sl.value, c_sl.value
    parab_line.set_ydata(a * xs**2 + b * xs + c)
    y_int.set_offsets([[0, c]])
    vx = -b / (2 * a) if a != 0 else 0.0
    vy = a * vx**2 + b * vx + c
    vertex.set_offsets([[vx, vy]])
    with info_out:
        info_out.clear_output(wait=True)
        print(f"y = {a:+.2f}·x² {b:+.2f}·x {c:+.2f}")
        print(f"vertex at x = {vx:.3f}, y = {vy:.3f}")
        print(f"y-intercept  = {c:+.3f}")
    fig.canvas.draw_idle()

for w in (a_sl, b_sl, c_sl): w.observe(redraw_parab, names="value")
redraw_parab()

display(VBox([HBox([a_sl, a_tx]), HBox([b_sl, b_tx]), HBox([c_sl, c_tx]), info_out]))

## 3 · Simulate the full free throw

No more 0.2 s clipping — we now want the full arc, from release to the hoop. Parameters: release speed 7.5 m/s at 52°, release height 2.10 m, hoop height 3.05 m, horizontal distance 4.57 m.

In [ ]:
v0, theta_deg = 7.5, 52.0
theta = np.radians(theta_deg)
v0x, v0y = v0 * np.cos(theta), v0 * np.sin(theta)
h0  = 2.10           # release height (m)
g   = 9.81

t = np.linspace(0, 2 * v0y / g + 0.1, 40)     # a bit past apex-descent
y_true  = h0 + v0y * t - 0.5 * g * t ** 2     # trajectory height vs time
y_noisy = y_true + np.random.normal(0, 0.03, size=t.shape)

fig2, ax2 = plt.subplots()
ax2.axhline(3.05, color="#27AE60", lw=1.2, linestyle=":", label="hoop rim  (3.05 m)")
ax2.scatter(t, y_noisy, s=30, color="#2C3E50", label="recorded height")
ax2.plot(t, y_true, color="#2E86C1", lw=1.2, alpha=0.7, label="true physics")
ax2.set_xlabel("time  t  (s)"); ax2.set_ylabel("height  y  (m)")
ax2.set_title("Free throw trajectory — height vs time")
ax2.legend()
plt.show()

## 4 · A line cannot fit this — confirm it

Before the trick, verify the claim. Fit $y = w\,t + b$ (Ch.1's model) and look at the residuals.

In [ ]:
linear = LinearRegression().fit(t.reshape(-1, 1), y_noisy)
y_lin  = linear.predict(t.reshape(-1, 1))
sse_lin = float(np.sum((y_lin - y_noisy) ** 2))

fig3, (axL, axR) = plt.subplots(1, 2, figsize=(10.5, 4.2))
axL.scatter(t, y_noisy, s=22, color="#2C3E50")
axL.plot(t, y_lin, color="#E74C3C", lw=2.0, label=f"line  (SSE = {sse_lin:.3f})")
axL.set_xlabel("t"); axL.set_ylabel("y"); axL.set_title("Best line — fails"); axL.legend()

axR.scatter(t, y_noisy - y_lin, s=22, color="#E74C3C")
axR.axhline(0, color="#7F8C8D", lw=0.7)
axR.set_xlabel("t"); axR.set_ylabel("residual  y − ŷ")
axR.set_title("Residuals show a clear curve → wrong model")
plt.tight_layout(); plt.show()

## 5 · Apply the feature-expansion trick

Engineer $x_1 = t$ and $x_2 = t^2$ and feed both to `LinearRegression`. We are still doing *linear* regression — on two features instead of one.

In [ ]:
poly  = PolynomialFeatures(degree=2, include_bias=False)
T     = poly.fit_transform(t.reshape(-1, 1))     # columns: [t, t^2]
model = LinearRegression().fit(T, y_noisy)
y_poly = model.predict(T)
sse_poly = float(np.sum((y_poly - y_noisy) ** 2))

w1, w2 = model.coef_
b_fit = model.intercept_
print(f"fitted w1 (~v0y)  : {w1:+.3f}   (physics: v0y = {v0y:+.3f})")
print(f"fitted w2 (~-g/2) : {w2:+.3f}   (physics: -g/2 = {-g/2:+.3f})")
print(f"fitted b  (~h0)   : {b_fit:+.3f}   (physics: h0  = {h0:+.3f})")
print(f"\nSSE: line = {sse_lin:.4f}   polynomial = {sse_poly:.4f}   (smaller = better)")

In [ ]:
fig4, ax4 = plt.subplots()
ax4.axhline(3.05, color="#27AE60", lw=1.0, linestyle=":", label="hoop rim")
ax4.scatter(t, y_noisy, s=22, color="#2C3E50", label="samples")
ax4.plot(t, y_lin,  color="#E74C3C", lw=1.8, label="line (Ch.1)")
ax4.plot(t, y_poly, color="#8E44AD", lw=2.2, label="parabola via linear regression")
ax4.set_xlabel("t"); ax4.set_ylabel("y"); ax4.legend()
ax4.set_title("Same linear-regression call, one extra feature, exact fit")
plt.show()

## 6 · Why it works — parabola in 1-D is a plane in 2-D

The parabola $y = 3x^2 - 2x + 1$ is curved. Plot it as the function $y(x_1, x_2) = 3 x_1 - 2 x_2 + 1$ — and it is *flat*. The curve only looks bent because we're seeing the subset where $x_1 = x_2^2$.

In [ ]:
from mpl_toolkits.mplot3d import Axes3D  # noqa: F401

x_line = np.linspace(-1.5, 1.5, 120)
curve_x1, curve_x2 = x_line**2, x_line
curve_y = 3 * curve_x1 - 2 * curve_x2 + 1

X1, X2 = np.meshgrid(np.linspace(0, 2.3, 20), np.linspace(-1.5, 1.5, 20))
Y_plane = 3 * X1 - 2 * X2 + 1

fig5 = plt.figure(figsize=(7.8, 5.4))
ax5 = fig5.add_subplot(111, projection="3d")
ax5.plot_surface(X1, X2, Y_plane, alpha=0.3, color="#8E44AD", edgecolor="none")
ax5.plot(curve_x1, curve_x2, curve_y, color="#2E86C1", lw=2.6,
         label="curve in 1-D")
ax5.set_xlabel("x1 = x²"); ax5.set_ylabel("x2 = x"); ax5.set_zlabel("y")
ax5.view_init(elev=22, azim=-52)
ax5.set_title("Same equation, flat plane in feature space")
ax5.legend()
plt.show()

## 7 · When the trick does **not** work

Feature expansion linearises models that are non-linear in the *input*. It does not help when the model is non-linear in the *parameter*. Compare two targets:

In [ ]:
x_plot = np.linspace(0, 3, 200)

# (a) y = a*x^2 + b*x + c — LINEAR in (a, b, c). Feature expansion works.
# (b) y = A * exp(k*x)    — NON-LINEAR in k. Cannot be turned into linear regression
#     by engineering features of x alone.

y_a = 2 * x_plot ** 2 - 3 * x_plot + 1
y_b = 1.5 * np.exp(0.8 * x_plot)

fig6, (axA, axB) = plt.subplots(1, 2, figsize=(10.2, 4.0))
axA.plot(x_plot, y_a, color="#8E44AD", lw=2.0)
axA.set_title("y = a·x² + b·x + c  (trick works)")
axA.set_xlabel("x"); axA.set_ylabel("y")
axB.plot(x_plot, y_b, color="#E74C3C", lw=2.0)
axB.set_title("y = A·exp(k·x)  (trick fails)")
axB.set_xlabel("x"); axB.set_ylabel("y")
plt.tight_layout(); plt.show()

print("Why the trick fails for y = A·exp(k·x):")
print("  k lives INSIDE the exponent. No substitution")
print("  of features φ(x) turns this into a linear-in-(A, k) model.")
print("  Either take log (only if y > 0) or use iterative optimisation (Ch.4).")

## 8 · Stretch exercise — overfit with too many features

Polynomial features are easy to abuse. Fit a degree-12 polynomial to the same 40 noisy samples and watch what happens.

In [ ]:
deg = 12
T_big = PolynomialFeatures(degree=deg, include_bias=False).fit_transform(t.reshape(-1, 1))
model_big = LinearRegression().fit(T_big, y_noisy)

t_dense = np.linspace(t.min(), t.max(), 400)
T_dense = PolynomialFeatures(degree=deg, include_bias=False).fit_transform(t_dense.reshape(-1, 1))
y_big  = model_big.predict(T_dense)

fig7, ax7 = plt.subplots()
ax7.scatter(t, y_noisy, s=22, color="#2C3E50", label="samples")
ax7.plot(t, y_poly, color="#8E44AD", lw=2.0, label="degree 2  (right)")
ax7.plot(t_dense, y_big, color="#E67E22", lw=1.6, label=f"degree {deg}  (over-fit)")
ax7.set_xlabel("t"); ax7.set_ylabel("y"); ax7.legend()
ax7.set_title("High-degree polynomials wiggle between the points")
plt.show()

print("Takeaway: 'more features' is not 'more accurate'.")
print("Match the model's capacity to the true signal — not to the noise.")

## 9 · Where this reappears

- **Pre-Req Ch.4.** When the optimal coefficients can't be solved in closed form, iterative small steps on the loss surface find them.
- **Pre-Req Ch.5.** The `LinearRegression.fit` call is literally $(\mathbf{X}^\top\mathbf{X})^{-1}\mathbf{X}^\top\mathbf{y}$ on the polynomial-feature matrix.
- **ML Ch.6 Regularisation.** Ridge and Lasso prevent the wild oscillation of §8.
- **ML Ch.4 Neural Networks.** Instead of hand-engineering features, learn them.